In [47]:
import requests
from bs4 import BeautifulSoup

In [48]:
url="https://pharmeasy.in/health-care?src=header"
r=requests.get(url)

In [49]:
with open("data/main.html","w",encoding="utf-8") as f:
    f.write(r.text)

In [50]:
with open("data/main.html","r") as f:
    html_docs=f.read()
soup=BeautifulSoup(html_docs,"html.parser")

In [51]:
links=[]
def get_links():
    for link in soup.find_all(class_="Category_categoryContainerLandingPageHome__151JB"):
         links.append(link.get("href"))
    return(links)

In [52]:
print(get_links())

['/health-care/top-products-9297', '/health-care/winter-store-16321', '/health-care/fitness-supplements-623', '/health-care/sexual-wellness-575', '/health-care/personal-care-877', '/health-care/health-condition-693', '/health-care/health-food-and-drinks-648', '/health-care/diabetic-care-145', '/health-care/cardiac-store-14374', '/health-care/ayurvedic-care-765', '/health-care/mother-and-baby-care-838', '/health-care/elderly-care-750', '/health-care/sports-nutrition-12931', '/health-care/healthcare-devices-717', '/health-care/skin-care-93', '/health-care/health-essentials-15393']


In [55]:
import os

def get_all_pages(links):
    os.makedirs("data",exist_ok=True)
    
    for i in range(len(links)):
        url=f"https://pharmeasy.in{links[i]}"
        r=requests.get(url)

        if r.status_code==200:
            with open(f"data/page{i}.html","w",encoding="utf-8") as f:
               f.write(r.text)
        else:
            print(f"Failed to fetch {url} | Status: {r.status_code}")

In [56]:
get_all_pages(links)

In [57]:
product_links={}

def product_links_extract():
    
    for i in range(16):
        with open(f"data/page{i}.html","r",encoding="utf-8") as f:
            html_docs=f.read()
            
        soup=BeautifulSoup(html_docs,"html.parser")
        key=f"product_{i}"
        value=[]
        
        for card in soup.find_all(class_="ProductCard_productCardWrapper__x7Ehg"):

            a_tag=card.find("a")
            if a_tag and a_tag.get("href"):
                value.append(a_tag.get("href"))
            
        product_links[key]=(value)
    return product_links

In [58]:
product_links_extract()

{'product_0': ['/health-care/products/pharmeasy-coenzyme-q10-coq10-200mg-with-piperine---for-heart-health-bottle-of-60-capsules-4315302',
  '/health-care/products/pharmeasy-diabetic-protein-powder-french-vanilla-flavour-400g-3963860',
  '/health-care/products/everherb-karela-jamun-juice---helps-maintains-healthy-sugar-levels--helps-in-weight-management---1l--501050',
  '/health-care/products/everherb-diabetic-care-juice-11-potent-ingredients--healthy-blood-sugar-management---1-litre-bottle-3046093',
  '/health-care/products/pharmeasy-pill-box-organizer-234116',
  '/health-care/products/pharmeasy-multivitamin-multimineral---pack-of-60-3491142',
  '/health-care/products/liveasy-foods-healthy-seed-mix-3515408',
  '/health-care/products/everherb-aloe-vera-juice-with-pulp---rejuvenates-skin-hair---1-litre-bottle-501051',
  '/health-care/products/everherb-amla-juice-rich-in-vitamin-c-natural-immunity-booster---1-litre-bottle-501052',
  '/health-care/products/everherb-pain-relief-oil---blend-

In [59]:
all_links=[]

for values in product_links.values():
    all_links.extend(values)

In [60]:
all_links

['/health-care/products/pharmeasy-coenzyme-q10-coq10-200mg-with-piperine---for-heart-health-bottle-of-60-capsules-4315302',
 '/health-care/products/pharmeasy-diabetic-protein-powder-french-vanilla-flavour-400g-3963860',
 '/health-care/products/everherb-karela-jamun-juice---helps-maintains-healthy-sugar-levels--helps-in-weight-management---1l--501050',
 '/health-care/products/everherb-diabetic-care-juice-11-potent-ingredients--healthy-blood-sugar-management---1-litre-bottle-3046093',
 '/health-care/products/pharmeasy-pill-box-organizer-234116',
 '/health-care/products/pharmeasy-multivitamin-multimineral---pack-of-60-3491142',
 '/health-care/products/liveasy-foods-healthy-seed-mix-3515408',
 '/health-care/products/everherb-aloe-vera-juice-with-pulp---rejuvenates-skin-hair---1-litre-bottle-501051',
 '/health-care/products/everherb-amla-juice-rich-in-vitamin-c-natural-immunity-booster---1-litre-bottle-501052',
 '/health-care/products/everherb-pain-relief-oil---blend-of-8-powerful-herbal-in

In [61]:
final_scrapped_values={
    "title":[],"category":[],"price":[],"no_of_ratings_given":[],"product_offer":[]
}

def scrapping_values(links):

    for i in range(len(links)):
        url=f"https://pharmeasy.in{links[i]}"
        r=requests.get(url)
        html_docs=r.text

        soup=BeautifulSoup(html_docs,"html.parser")

        #title
        h1=soup.h1
        final_scrapped_values["title"].append(h1.get_text(strip=True) if h1 else None)

        # category
        a_tag=soup.find_all("a",class_="BreadCrumbLink_breadCrumb___OJ18")
        final_scrapped_values["category"].append(a_tag[1].get_text(strip=True) if len(a_tag)>1 else None)

        # price
        price_tag=soup.find("div",class_="ProductPriceContainer_mrp___GtBV")
        final_scrapped_values["price"].append(price_tag.get_text(strip=True) if price_tag else None)

        #ratings
        rating_tag=soup.find("div",class_="OverviewSection_ratingText__qUpG6")
        final_scrapped_values["no_of_ratings_given"].append(rating_tag.get_text(strip=True) if rating_tag else None)

        #offers
        offer=soup.find("div",class_="ProductPriceContainer_discountContainer__5arLQ")
        final_scrapped_values["product_offer"].append(offer.get_text(strip=True) if offer else None)

    return final_scrapped_values

In [62]:
data=scrapping_values(all_links)

In [63]:
import pandas as pd

In [64]:
df=pd.DataFrame(data)

In [65]:
df

,title,category,price,no_of_ratings_given,product_offer
0,Pharmeasy Coenzyme Q10 (Coq10) 200mg With Pipe...,Healthcare,₹770.65,None,53% OFF
1,Pharmeasy Diabetic Protein Powder French Vanil...,Healthcare,₹382.27,(60ratings),57% OFF
2,Everherb Karela Jamun Juice - Helps Maintains ...,Healthcare,₹160.85,(2174ratings),57% OFF
3,Everherb Diabetic Care Juice-13 Potent Herbs -...,Healthcare,₹230.42,(1429ratings),59% OFF
4,Pharmeasy Pill Box Organizer,Healthcare,₹234.53,(905ratings),53% OFF
...,...,...,...,...,...
315,Hk Vitals Biotin (60 Tablets) | Supplement For...,Healthcare,₹361.13,(6ratings),23% OFF
316,Accusure Fetal Doppler For Doctors And Mothers...,Healthcare,₹2001,(12ratings),71% OFF
317,Dr. Biswas Good Health Strong Capsule (Pack Of 2),Healthcare,₹211.2,(31ratings),56% OFF
318,Kapiva Himalayan Shilajit/Shilajeet Resin 40g ...,Healthcare,₹1648.9,None,45% OFF


In [66]:
df.to_csv("myData.csv",index=False)

In [67]:
pharmeasy= {
    "title": [...],
    "category": [...],
    "price": [...],
    "no_of_ratings_given": [...],
    "product_offer": [...]
}

In [68]:
def scrape_all_pages():

    for page_index in range(16):

        file_path = f"data/page{page_index}.html"

        if not os.path.exists(file_path):
            print(f"Missing file: {file_path}")
            continue

        with open(file_path, "r", encoding="utf-8") as f:
            html_docs = f.read()

        soup = BeautifulSoup(html_docs, "html.parser")

        #Extract category
        breadcrumbs = soup.find_all("a", class_="BreadCrumbLink_breadCrumb___OJ18")
        category = breadcrumbs[2].get_text(strip=True) if len(breadcrumbs) > 2 else None

        #Product Links
        product_links = []

        for card in soup.find_all(class_="ProductCard_productCardWrapper__x7Ehg"):
            a_tag = card.find("a")
            if a_tag and a_tag.get("href"):
                product_links.append(a_tag.get("href"))

        # Visiting product
        for link in product_links:

            product_url = "https://pharmeasy.in" + link
            r = requests.get(product_url)

            if r.status_code != 200:
                continue

            product_soup = BeautifulSoup(r.text, "html.parser")

            # title
            h1 = product_soup.h1
            pharmeasy["title"].append(
                h1.get_text(strip=True) if h1 else None
            )

            # category
            pharmeasy["category"].append(category)

            # price
            price_tag = product_soup.find(
                "div", class_="ProductPriceContainer_mrp___GtBV"
            )
            pharmeasy["price"].append(
                price_tag.get_text(strip=True) if price_tag else None
            )

            # ratings
            rating_tag = product_soup.find(
                "div", class_="OverviewSection_ratingText__qUpG6"
            )
            pharmeasy["no_of_ratings_given"].append(
                rating_tag.get_text(strip=True) if rating_tag else None
            )

            # offer
            offer_tag = product_soup.find(
                "div", class_="ProductPriceContainer_discountContainer__5arLQ"
            )
            pharmeasy["product_offer"].append(
                offer_tag.get_text(strip=True) if offer_tag else None
            )

    return pharmeasy

In [69]:
pharmeasy_data=scrape_all_pages()

In [70]:
df1=pd.DataFrame(pharmeasy)

In [71]:
df1.to_csv("pharmeasy.csv",index=False)

In [72]:
df1

,title,category,price,no_of_ratings_given,product_offer
0,Ellipsis,Ellipsis,Ellipsis,Ellipsis,Ellipsis
1,Pharmeasy Coenzyme Q10 (Coq10) 200mg With Pipe...,Must Haves,₹770.65,None,53% OFF
2,Pharmeasy Diabetic Protein Powder French Vanil...,Must Haves,₹382.27,(60ratings),57% OFF
3,Everherb Karela Jamun Juice - Helps Maintains ...,Must Haves,₹160.85,(2174ratings),57% OFF
4,Everherb Diabetic Care Juice-13 Potent Herbs -...,Must Haves,₹230.42,(1429ratings),59% OFF
...,...,...,...,...,...
316,Hk Vitals Biotin (60 Tablets) | Supplement For...,Explore More,₹361.13,(6ratings),23% OFF
317,Accusure Fetal Doppler For Doctors And Mothers...,Explore More,₹2001,(12ratings),71% OFF
318,Dr. Biswas Good Health Strong Capsule (Pack Of 2),Explore More,₹211.2,(31ratings),56% OFF
319,Kapiva Himalayan Shilajit/Shilajeet Resin 40g ...,Explore More,₹1648.9,None,45% OFF


In [73]:
df1.drop(0,inplace=True)

In [74]:
df1.reset_index(drop=True,inplace=True)

In [75]:
df1

,title,category,price,no_of_ratings_given,product_offer
0,Pharmeasy Coenzyme Q10 (Coq10) 200mg With Pipe...,Must Haves,₹770.65,None,53% OFF
1,Pharmeasy Diabetic Protein Powder French Vanil...,Must Haves,₹382.27,(60ratings),57% OFF
2,Everherb Karela Jamun Juice - Helps Maintains ...,Must Haves,₹160.85,(2174ratings),57% OFF
3,Everherb Diabetic Care Juice-13 Potent Herbs -...,Must Haves,₹230.42,(1429ratings),59% OFF
4,Pharmeasy Pill Box Organizer,Must Haves,₹234.53,(905ratings),53% OFF
...,...,...,...,...,...
315,Hk Vitals Biotin (60 Tablets) | Supplement For...,Explore More,₹361.13,(6ratings),23% OFF
316,Accusure Fetal Doppler For Doctors And Mothers...,Explore More,₹2001,(12ratings),71% OFF
317,Dr. Biswas Good Health Strong Capsule (Pack Of 2),Explore More,₹211.2,(31ratings),56% OFF
318,Kapiva Himalayan Shilajit/Shilajeet Resin 40g ...,Explore More,₹1648.9,None,45% OFF


In [76]:
df1.to_csv("pharmeasy.csv", index=False)